# 06 — Robustness of the Long-Horizon Return Signal

The non-overlapping test confirmed the 120-day return signal is **real and significant**
(honest IC ~0.10, p=0.027, positive in 22/29 independent periods). Before committing to
paid fundamentals, one last free check: **is the edge broad-based and regime-robust, or
is it a hidden concentration bet?**

A significant aggregate IC can still hide two failure modes that would change the
decision:

1. **Sector concentration** — if the signal lives almost entirely in one sector (say
   tech), it is really a sector bet, not a general cross-sectional edge, and fundamentals
   might not generalize.
2. **Regime concentration** — if the aggregate IC is carried by one or two extreme periods
   (e.g. the 2020 rebound), it is fragile. In particular, did it survive 2022, the
   momentum-crash regime where price signals inverted?

This notebook breaks the 120-day IC down by **sector** and by **year**, and checks how
many individual names carry it. If the edge is diversified across sectors, positive in
most years including 2022, and not driven by a handful of tickers, the fundamentals bet
rests on solid ground.

In [ ]:
from pathlib import Path
import sys, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.grid"] = True; plt.rcParams["grid.alpha"] = 0.3
warnings.filterwarnings("ignore")

In [ ]:
PROJECT_ROOT = Path.cwd().resolve()
for parent in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (parent / 'src').is_dir():
        PROJECT_ROOT = parent; break
else:
    raise RuntimeError('Run from inside the StockForecastRisk repository.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.forecast_engine.features.schema import FEATURE_NAMES, NON_FEATURE_COLUMNS
from src.forecast_engine.data.loader import load_processed_features

data = load_processed_features()
data["date"] = pd.to_datetime(data["date"])
data = data.sort_values(["symbol", "date"]).reset_index(drop=True)

NON_STATIONARY_LEVELS = ["sma_10","sma_20","sma_50","sma_200","ema_12","ema_26","vwap_20"]
SI = ["short_interest","short_interest_change","days_to_cover","days_to_cover_change"]
EXCLUDE = set(NON_FEATURE_COLUMNS) | set(NON_STATIONARY_LEVELS) | set(SI) | {"short_history","history_rows"}
feature_cols = [c for c in FEATURE_NAMES if c not in EXCLUDE and c in data.columns and not data[c].isna().all()]

HORIZON = 120
data[f"fwd_ret_{HORIZON}"] = data.groupby("symbol", group_keys=False).apply(
    lambda g: np.log(g["adj_close"].astype(float).shift(-HORIZON) / g["adj_close"].astype(float)))
print(f"features: {len(feature_cols)}, horizon: {HORIZON}d")

## Generate out-of-sample predictions (Ridge, purged walk-forward)

Same setup as the significance check: one out-of-sample Ridge prediction per row, so
the IC breakdowns below are all on genuinely held-out predictions.

In [ ]:
N_SPLITS = 5
target = f"fwd_ret_{HORIZON}"
md_h = data.dropna(subset=feature_cols + [target]).reset_index(drop=True)

# carry sector for the breakdown
sector_col = "gics_sector" if "gics_sector" in md_h.columns else None
X = md_h[feature_cols].to_numpy(np.float32)
y = md_h[target].to_numpy(np.float32)

dates = np.sort(md_h["date"].unique())
edges = np.array_split(dates, N_SPLITS + 1)
preds = np.full(len(md_h), np.nan, np.float32)
for f in range(N_SPLITS):
    cutoff = edges[f][-1] - pd.Timedelta(days=HORIZON * 2)
    tr = md_h.index[md_h["date"] <= cutoff]
    te = md_h.index[md_h["date"].isin(edges[f + 1])]
    if len(tr) and len(te):
        sc = StandardScaler().fit(X[tr])
        preds[te] = Ridge(alpha=1.0).fit(sc.transform(X[tr]), y[tr]).predict(sc.transform(X[te]))

md_h["pred"] = preds
oos = md_h.dropna(subset=["pred"]).copy()
oos["year"] = oos["date"].dt.year
print(f"out-of-sample predictions: {len(oos):,} rows, "
      f"{oos['date'].min().date()} -> {oos['date'].max().date()}")

## 1. IC by year — is it consistent, and did it survive 2022?

Per-date cross-sectional IC averaged within each year. The key cells to read are 2020
(did one rebound year carry it?) and 2022 (did it survive the momentum crash?).

In [ ]:
def per_date_ic(frame):
    recs = {}
    for d, g in frame.groupby("date"):
        if g[target].nunique() > 2 and g["pred"].nunique() > 2:
            c = spearmanr(g["pred"], g[target]).correlation
            if np.isfinite(c): recs[d] = c
    return pd.Series(recs)

ic_by_date = per_date_ic(oos)
ic_by_date.index = pd.to_datetime(ic_by_date.index)
ic_year = ic_by_date.groupby(ic_by_date.index.year).mean()

print("120-day IC by year:")
display(ic_year.round(4).to_frame("mean_ic"))

fig, ax = plt.subplots(figsize=(12, 4))
colors = ["tab:green" if v > 0 else "tab:red" for v in ic_year.values]
ax.bar(ic_year.index.astype(str), ic_year.values, color=colors, alpha=0.85)
ax.axhline(0, color="black", lw=0.8)
ax.axhline(ic_year.mean(), color="blue", ls="--", label=f"mean {ic_year.mean():+.3f}")
ax.set_title("120-day return IC by year — consistent, or one-regime?")
ax.set_ylabel("mean rank IC"); ax.legend()
n_pos = (ic_year > 0).sum()
ax.text(0.02, 0.95, f"{n_pos}/{len(ic_year)} years positive\n2022: {ic_year.get(2022, float('nan')):+.3f}",
        transform=ax.transAxes, va="top",
        bbox=dict(boxstyle="round", fc="white", alpha=0.85))
plt.tight_layout(); plt.show()

## 2. IC by sector — broad-based, or one sector?

Pool predictions within each GICS sector and compute the IC there. A signal concentrated
in one or two sectors is really a sector bet; a signal present across most sectors is a
general cross-sectional edge that fundamentals could plausibly extend.

In [ ]:
if sector_col:
    rows = []
    for sec, g in oos.groupby(sector_col):
        # per-date IC within the sector, then average
        s = per_date_ic(g)
        rows.append({"sector": sec, "mean_ic": s.mean() if len(s) else np.nan,
                     "n_rows": len(g), "n_names": g["symbol"].nunique()})
    sector_ic = pd.DataFrame(rows).dropna().sort_values("mean_ic", ascending=False).reset_index(drop=True)
    display(sector_ic.round(4))

    fig, ax = plt.subplots(figsize=(11, 5))
    colors = ["tab:green" if v > 0 else "tab:red" for v in sector_ic["mean_ic"]]
    ax.barh(sector_ic["sector"], sector_ic["mean_ic"], color=colors, alpha=0.85)
    ax.axvline(0, color="black", lw=0.8)
    ax.axvline(ic_by_date.mean(), color="blue", ls="--", label=f"overall {ic_by_date.mean():+.3f}")
    ax.set_title("120-day return IC by GICS sector — broad-based or concentrated?")
    ax.set_xlabel("mean rank IC"); ax.legend()
    n_pos = (sector_ic["mean_ic"] > 0).sum()
    ax.text(0.98, 0.05, f"{n_pos}/{len(sector_ic)} sectors positive",
            transform=ax.transAxes, ha="right",
            bbox=dict(boxstyle="round", fc="white", alpha=0.85))
    plt.tight_layout(); plt.show()
else:
    print("No gics_sector column available; skipping sector breakdown.")
    sector_ic = None

## 3. Per-name contribution — a few tickers, or many?

For each ticker, the correlation between its predictions and its realised 120-day returns.
If the aggregate signal comes from a handful of names, the distribution will be dominated
by a few large positives; if it is broad, most names will lean mildly positive.

In [ ]:
name_rows = []
for sym, g in oos.groupby("symbol"):
    if g[target].nunique() > 5 and g["pred"].nunique() > 5:
        c = spearmanr(g["pred"], g[target]).correlation
        if np.isfinite(c):
            name_rows.append({"symbol": sym, "ic": c, "n": len(g)})
name_ic = pd.DataFrame(name_rows)

frac_pos = (name_ic["ic"] > 0).mean()
print(f"per-name IC: {len(name_ic)} tickers, {frac_pos:.0%} positive, "
      f"median {name_ic['ic'].median():+.3f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(name_ic["ic"], bins=50, color="tab:blue", alpha=0.8)
ax.axvline(0, color="red", ls="--")
ax.axvline(name_ic["ic"].median(), color="blue", lw=2, label=f"median {name_ic['ic'].median():+.3f}")
ax.set_title(f"Per-ticker IC distribution ({frac_pos:.0%} of names positive)")
ax.set_xlabel("per-ticker rank IC"); ax.legend()
plt.tight_layout(); plt.show()

print("\nMost positive names:")
display(name_ic.sort_values("ic", ascending=False).head(8).round(3))
print("Most negative names:")
display(name_ic.sort_values("ic").head(8).round(3))

## Conclusion — is the signal diversified enough to build on?

Fill from the numbers above.

- **By year:** ___ / ___ years positive. **2022 specifically: ____** — survived the
  momentum crash, or not?
- **By sector:** ___ / ___ sectors positive. Concentrated in one, or spread across many?
- **By name:** ___% of tickers positive, median ____. Broad, or a few names?

### The read

- **Broad across sectors, positive in most years (including 2022), most names leaning
  positive** → the edge is a genuine, diversified cross-sectional signal, not a hidden
  concentration or single-regime bet. The fundamentals investment rests on solid ground:
  proceed to Sharadar and A/B fundamentals at 120-day against the Ridge baseline.

- **Concentrated in one sector, or carried by one/two years, or driven by a handful of
  names** → the aggregate IC is real but fragile. Treat the return signal cautiously,
  and be more conservative about the fundamentals spend — the edge may not generalize the
  way a diversified signal would.

This is the final free check. Whatever it shows, the return signal has now been tested
for reality (significance), independence (non-overlapping), and robustness (here) — the
full discipline before any money is committed.